# Sharks — Análisis de incidentes y propuesta para escuelas de surf

Estudiamos 629 incidentes relacionados con surf registrados en USA y Australia entre 2000 y 2025, ambos incluidos: 26 años.

**Objetivo:**
Analizamos dónde y cuándo se concentran los incidentes y qué diferencias presentan en mortalidad. Utilizamos estos resultados para orientar un servicio de información y formación para escuelas y centros de surf.

Queremos identificar zonas donde iniciar la propuesta, periodos en los que reforzar la comunicación y contenidos adaptados al contexto de cada territorio.

**Hipótesis:** 

Planteamos que los incidentes presentan diferencias geográficas, estacionales y de mortalidad que justifican adaptar la información y la preparación de las escuelas a cada territorio.

También comprobamos si el verano concentra más incidentes dentro de cada país.

### Cómo organizamos el notebook

1. Cargamos los datos y seleccionamos el periodo y los países.
2. Limpiamos las categorías, los valores ausentes, las horas y las fechas.
3. Comparamos países, estados, horarios, estaciones, perfiles y especies.
4. Relacionamos los resultados con nuestra propuesta de servicio.

Ejecutamos las celdas en orden, con `tiburones.csv` en la misma carpeta.

### Alcance y decisiones

- Trabajamos con una tabla final de 629 registros y 12 columnas.
- Analizamos las categorías finales `surfing` y `unprovoked`.
- Utilizamos `life = 1` para supervivientes y `life = 0` para fallecidos.
- Realizamos la comparación geográfica por país y estado.
- Aplicamos distintas estaciones a USA y Australia según su hemisferio.
- Incluimos los valores asignados durante la limpieza y explicamos esas decisiones en sus apartados.
- Interpretamos los porcentajes como la distribución y la mortalidad de los incidentes analizados.

## 1. Carga y selección de datos

### 1.1. Leer el archivo

Cargamos el CSV y revisamos las primeras filas, el tamaño y los tipos de datos. Así conocemos la estructura antes de modificarla.

In [186]:
import pandas as pd

sharks_df = pd.read_csv("tiburones.csv")

In [187]:
sharks_df.head()

,Date,Year,Type,Country,State,Location,Activity,Name,Sex,Age,...,Species,Source,pdf,href formula,href,Case Number,Case Number.1,original order,Unnamed: 21,Unnamed: 22
0,18th September,2026,Unprovoked,Australia,Western Australia,Sorrento Beach Perth,Swimming,Greg O'Neil,M,63,...,Great White Shark 6m (20ft),Keith Cowley: Kevin McMurray Trackingsharks.co...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,16th September,2026,Unprovoked,Canada,Quebec,Off the coast of Perce Le Bilbo dive site,Diving,Unknown Male,M,?,...,Great White Shark,Keith Cowley: Kevin McMurray Trackingsharks.co...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,14th September,2026,Unprovoked,Bahamas,Bimini,Bimini Island,Swimming,Unknown Austrian Tourist,F,37,...,Unknown,Kevin McMurray Trackingsharks.com: Keith Cowley,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,13th September,2026,Unprovoked,Australia,Western Australia,Geraldton,Surfing,Mel Ismail,M,50's,...,Unknown,Simon De Marchi,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,7th September,2026,Unprovoked,USA,Hawaii,Honolulu,Surfing,Wants to remain anonymous,M,24,...,Tiger Shark suspected,Keith Cowley,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [188]:
print(sharks_df.shape)

(7125, 23)


In [189]:
sharks_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7125 entries, 0 to 7124
Data columns (total 23 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Date            7125 non-null   str    
 1   Year            7123 non-null   str    
 2   Type            7107 non-null   str    
 3   Country         7075 non-null   str    
 4   State           6638 non-null   str    
 5   Location        6558 non-null   str    
 6   Activity        6542 non-null   str    
 7   Name            6907 non-null   str    
 8   Sex             6547 non-null   str    
 9   Age             4131 non-null   str    
 10  Injury          7089 non-null   str    
 11  Fatal Y/N       6564 non-null   str    
 12  Time            3598 non-null   str    
 13  Species         3994 non-null   str    
 14  Source          7105 non-null   str    
 15  pdf             6799 non-null   str    
 16  href formula    6794 non-null   str    
 17  href            6796 non-null   str    
 18 

### 1.2. Retirar columnas auxiliares

Eliminamos el bloque de columnas desde `pdf` hasta `Unnamed: 22`. Son referencias y campos auxiliares que no utilizamos en las mediciones.

**Comprobación:** revisamos las columnas restantes con `.info()`. El borrado con `inplace=True` modifica directamente `sharks_df`.

In [190]:
sharks_df.drop(columns=sharks_df.loc[:, "pdf":"Unnamed: 22"].columns, inplace=True)

In [191]:
sharks_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7125 entries, 0 to 7124
Data columns (total 15 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   Date       7125 non-null   str  
 1   Year       7123 non-null   str  
 2   Type       7107 non-null   str  
 3   Country    7075 non-null   str  
 4   State      6638 non-null   str  
 5   Location   6558 non-null   str  
 6   Activity   6542 non-null   str  
 7   Name       6907 non-null   str  
 8   Sex        6547 non-null   str  
 9   Age        4131 non-null   str  
 10  Injury     7089 non-null   str  
 11  Fatal Y/N  6564 non-null   str  
 12  Time       3598 non-null   str  
 13  Species    3994 non-null   str  
 14  Source     7105 non-null   str  
dtypes: str(15)
memory usage: 2.0 MB


### 1.3. Reducir las variables

Retiramos `Name`, `Injury`, `Source` y `Location` para trabajar con una tabla más sencilla.

Al eliminar `Location`, renunciamos al análisis por localidad en esta versión. Conservamos el estado para la comparación geográfica.

In [192]:
sharks_df.drop(columns=["Name", "Injury", "Source","Location"], inplace=True)

### 1.4. Preparar el año

Revisamos el tipo de `Year` antes de filtrar el periodo de estudio.

In [193]:
sharks_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7125 entries, 0 to 7124
Data columns (total 11 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   Date       7125 non-null   str  
 1   Year       7123 non-null   str  
 2   Type       7107 non-null   str  
 3   Country    7075 non-null   str  
 4   State      6638 non-null   str  
 5   Activity   6542 non-null   str  
 6   Sex        6547 non-null   str  
 7   Age        4131 non-null   str  
 8   Fatal Y/N  6564 non-null   str  
 9   Time       3598 non-null   str  
 10  Species    3994 non-null   str  
dtypes: str(11)
memory usage: 1.1 MB


Contamos los nulos y convertimos `Year` a número con `pd.to_numeric()`.

`errors="coerce"` convierte los textos no interpretables en nulos. Después seleccionamos los años de **2000 a 2025**; las filas sin año válido quedan fuera.

**A revisar:** textos como `2015,00` necesitan normalizar la coma antes de esta conversión para no perder registros válidos.

In [194]:
sharks_df["Year"].isnull().sum()

np.int64(2)

In [195]:
sharks_df["Year"] = pd.to_numeric(sharks_df["Year"], errors="coerce")

In [196]:
sharks_df = sharks_df.loc[(sharks_df["Year"] >= 2000) & (sharks_df["Year"] <= 2025)]

### 1.5. Tipo de incidente

Revisamos las etiquetas y seleccionamos `unprovoked`.

**Decisión del código:** `?` y `Questionable` también se convierten en `unprovoked`. Esto amplía la selección con casos dudosos; no es solo una corrección de formato.

In [197]:
sharks_df["Type"].isnull().sum()
sharks_df["Type"].unique()

<ArrowStringArray>
[         'Unprovoked',            'Provoked',        'Questionable',
          'unprovoked',           ' Provoked',          'Watercraft',
        'Sea Disaster',                   nan,                   '?',
         'Unconfirmed',          'Unverified',             'Invalid',
 'Under investigation']
Length: 13, dtype: str

In [198]:
sharks_df["Type"] = sharks_df["Type"].replace({"Unprovoked" : "unprovoked", "?" : "unprovoked", "Questionable" : "unprovoked"})
sharks_df["Type"].unique()

<ArrowStringArray>
[         'unprovoked',            'Provoked',           ' Provoked',
          'Watercraft',        'Sea Disaster',                   nan,
         'Unconfirmed',          'Unverified',             'Invalid',
 'Under investigation']
Length: 10, dtype: str

In [199]:
sharks_df = sharks_df.loc[(sharks_df["Type"] == "unprovoked")]


### 1.6. Unificar los países

Pasamos `Country` a minúsculas para que `USA` y `usa`, por ejemplo, se reconozcan como el mismo valor. Esto unifica etiquetas; no elimina incidentes.

In [200]:
sharks_df["Country"] = sharks_df["Country"].str.lower()
sharks_df["Country"].unique()

<ArrowStringArray>
[                           'mozambique',
                             'australia',
                                   'usa',
                      'french polynesia',
                                 'samoa',
                              'columbia',
                               'bahamas',
                           'puerto rico',
                                 'spain',
                        'canary islands',
                          'south africa',
                               'vanuatu',
                               'jamaica',
                                'israel',
                                'mexico',
                              'maldives',
                           'philippines',
                      'turks and caicos',
                         'new caledonia',
                                 'egypt',
                              'thailand',
                           'new zealand',
                                'hawaii',
               

In [201]:
sharks_df.head()

,Date,Year,Type,Country,State,Activity,Sex,Age,Fatal Y/N,Time,Species
60,25th December,2025.0,unprovoked,mozambique,Inhambe Province,Not stated,?,?,Y,?,Unknown
61,21st December,2025.0,unprovoked,australia,Western Australia,SCUBA Diving,M,?,N,?,Great White Shark
62,21st December,2025.0,unprovoked,usa,California,Swimming,F,55,Y,1200hrs,Great White Shark
63,12th December,2025.0,unprovoked,usa,Sonoma County California,Surfing,M,?,N,0800hrs,Suspected Great White Shark
65,27th November,2025.0,unprovoked,australia,NSW,Swimming,M,26,N,0630hrs,3m Bull shark


### 1.7. Seleccionar USA y Australia

Usamos `.isin()` para seleccionar ambos países y guardamos el subconjunto en `sharks_df_usa_australia`.

In [202]:
condition = (sharks_df["Country"].isin(["usa", "australia"]))
condition

60      False
61       True
62       True
63       True
65       True
        ...  
2865     True
2866    False
2868    False
2869     True
2871    False
Name: Country, Length: 2245, dtype: bool

In [203]:
sharks_df_usa_australia = sharks_df[condition]

## 2. Limpieza de las variables

### 2.1. Estados

Unificamos mayúsculas, espacios, abreviaturas y errores para no contar una misma zona con varios nombres.

La función también convierte algunas localidades a su estado, como Los Angeles a California.

In [204]:
sharks_df_usa_australia["State"] = sharks_df_usa_australia["State"].str.lower()
sharks_df_usa_australia["State"].unique()

<ArrowStringArray>
[                   'western australia',
                           'california',
             'sonoma county california',
                                  'nsw',
                               'hawaii',
                                'texas',
                           'queensland',
                      'south australia',
                       'off california',
                              'florida',
                       'long island ny',
                       'south carolina',
                       'north carolina',
                                   'wa',
                             'victoria',
                               'floria',
                            'galveston',
                     'new  south wales',
                      'new south wales',
                    'westerm australia',
                             'new york',
                           'new jersey',
                       'new south ales',
                                'samoa

**Cómo funciona Regex:** `\s+` detecta espacios repetidos, `^` y `$` delimitan el texto completo y `|` indica alternativas. Las reglas explícitas evitan agrupar nombres solo por parecido.

In [205]:
import re

def clean_state(value):
    # Conservar los valores nulos.
    if not isinstance(value, str):
        return value

    # Pasar a minúsculas y quitar espacios sobrantes.
    value = value.lower().strip()
    value = re.sub(r"\s+", " ", value)

    # Unificar nombres conocidos.
    replacements = {
        r"^(nsw|new south ales)$": "new south wales",
        r"^westerm australia$": "western australia",
        r"^noirth carolina$": "north carolina",
        r"^floria$": "florida",

        r"^(sonoma county california|off california|los angeles)$": "california",
        r"^franklin county, florida$": "florida",
        r"^long island ny$": "new york",
        r"^galveston$": "texas",
        r"^maui$": "hawaii",
        r"^wa$": "western australia",
        r"^virgin islands$": "us virgin islands"}

    for pattern, replacement in replacements.items():
        value = re.sub(pattern, replacement, value)

    return value

In [206]:
sharks_df_usa_australia["State"] = sharks_df_usa_australia["State"].apply(clean_state)
sharks_df_usa_australia["State"].unique()


<ArrowStringArray>
[                   'western australia',
                           'california',
                      'new south wales',
                               'hawaii',
                                'texas',
                           'queensland',
                      'south australia',
                              'florida',
                             'new york',
                       'south carolina',
                       'north carolina',
                             'victoria',
                           'new jersey',
                                'samoa',
                            'louisiana',
                                    nan,
                              'alabama',
                             'maryland',
                              'georgia',
                               'oregon',
                    'us virgin islands',
                                'maine',
                              'bahamas',
                             'tasmania

### 2.2. Actividades

Definimos reglas para corregir la escritura y agrupar los textos que contienen `surf` como `surfing`.

Esta categoría prevista incluye modalidades distintas y menciones a tablas. La función también asigna los nulos a `surfing`.

In [207]:
sharks_df_usa_australia["Activity"] = sharks_df_usa_australia["Activity"].str.lower()


In [208]:
import re

def clean_activity(value):
    if pd.isna(value):
        return "surfing"
    if not isinstance(value, str):
        return value

    # Unificar mayúsculas, espacios y guiones.
    value = value.lower().strip()
    value = re.sub(r"-", " ", value)
    value = re.sub(r"\s+", " ", value)

    # Corregir errores y unificar nombres equivalentes.
    replacements = {
        r"\bfihing\b": "fishing",
        r"\bkakaying\b": "kayaking",
        r"\bsurfng\b": "surfing",
        r"\bswimmingq\b": "swimming",

        r"\b(bodyboarding|body boarding|boggie boarding|boogie boarding)\b": "bodyboarding",
        r"\b(body surfing|bodysurfing)\b": "bodysurfing",

        r"\b(stand up paddle boarding|stand up paddleboarding|paddle boarding|paddleboarding|sup)\b": "paddleboarding",

        r"\b(kite boarding|kiteboarding|kite surfing|kitesurfing)\b": "kitesurfing",

        r"\b(surf sking|surf skiing)\b": "surf skiing",
        r"\bfree diving\b": "freediving",

        r"^lifeguard exercises$": "lifeguard training",
        r"^lifeguard training exercise$": "lifeguard training",

        r"^jumped into (the )?water$": "jumping into water",
        r"^diving for abalone$": "abalone diving",

        r"^(not stated|undisclosed)$": "unknown"
    }

    for pattern, replacement in replacements.items():
        value = re.sub(pattern, replacement, value)

    # Dar el mismo formato a las actividades combinadas.
    value = re.sub(r"\s*/\s*", " / ", value)
    if re.search(r"surf", value):
        value = "surfing"

    return value


In [209]:
sharks_df_usa_australia["Activity"] = sharks_df_usa_australia["Activity"].apply(clean_state)
sharks_df_usa_australia["Activity"].unique()

<ArrowStringArray>
[                                    'scuba diving',
                                         'swimming',
                                          'surfing',
                                    'foil boarding',
                                          'fishing',
                                 'fishing/swimming',
                                       'snorkeling',
                                         'kayaking',
                                           'diving',
                                      'undisclosed',
 ...
            'floating face-down in knee-deep water',
                     'standing alongside surfboard',
                       'diving (shell maintenance)',
                            'swimming / snorkeling',
                          'swimming / body surfing',
                        'swimming out to porpoises',
            'windsurfing, but sitting on his board',
                                 'surfing / wading',
 'spearfishing, holdin

In [210]:
print(sharks_df_usa_australia["Activity"])

61      scuba diving
62          swimming
63           surfing
65          swimming
66          swimming
            ...     
2859         surfing
2863        swimming
2864    spearfishing
2865             NaN
2869         surfing
Name: Activity, Length: 1575, dtype: str


In [211]:
sharks_df_usa_australia["Activity"].value_counts()

Activity
surfing                                             629
swimming                                            257
wading                                               81
spearfishing                                         70
snorkeling                                           59
                                                   ... 
swimming out to porpoises                             1
windsurfing, but sitting on his board                 1
surfing / wading                                      1
spearfishing, holding mesh bag with speared fish      1
boogie boarding / wading                              1
Name: count, Length: 160, dtype: int64

In [212]:
sharks_activity = sharks_df_usa_australia.loc[sharks_df_usa_australia["Activity"] == "surfing"]

**Resultado del filtro:** conservamos las filas que en este punto tienen el valor exacto `surfing`.

In [213]:
sharks_activity.head()

,Date,Year,Type,Country,State,Activity,Sex,Age,Fatal Y/N,Time,Species
63,12th December,2025.0,unprovoked,usa,california,surfing,M,?,N,0800hrs,Suspected Great White Shark
74,7th October,2025.0,unprovoked,australia,south australia,surfing,M,50+,N,1330hrs,Bronze whaler?
77,6th September,2025.0,unprovoked,australia,new south wales,surfing,M,57,Y,0930hrs,Great White Shark
80,18th August,2025.0,unprovoked,australia,new south wales,surfing,M,?,N,0730hrs,5m (16.5ft) Great White
83,7th August,2025.0,unprovoked,australia,new south wales,surfing,M,9,N,1630hrs,Suspected Great White


### 2.3. Sexo

Unificamos las etiquetas en `male` y `female` para agruparlas después.

In [214]:
print(sharks_activity["Sex"])

63      M
74      M
77      M
80      M
83      M
       ..
2825    M
2843    M
2857    M
2859    M
2869    M
Name: Sex, Length: 629, dtype: str


In [215]:
sharks_activity["Sex"] = sharks_activity["Sex"].replace({"lli" : "male", "M": "male",'M ': "male",' M': "male",'m': "male",'?': "male", "F" : "female", "F " : "female", "NaN" :"male" })


In [216]:
sharks_activity["Sex"].isna().sum()

np.int64(9)

In [217]:
sharks_activity["Sex"] = sharks_activity["Sex"].fillna("male")

**Comprobación:** revisamos la columna después de guardar los cambios.

### 2.4. Edad

Convertimos `Age` a número. Los textos no interpretables pasan a nulos y se rellenan con la media de las edades disponibles.

In [218]:
sharks_activity["Age"] = pd.to_numeric(sharks_activity["Age"], errors="coerce")


In [219]:
sharks_activity["Age"] = sharks_activity["Age"].fillna(sharks_activity["Age"].mean()).astype(int)

**Resultado:** la edad queda numérica y sin nulos si existe una media válida. `.astype(int)` elimina los decimales; no redondea al entero más cercano.

### 2.5. Supervivencia

Transformamos las etiquetas en números para calcular recuentos y proporciones:

- `Y → 0`: fallecido.
- `N → 1`: superviviente.
- `Nq`, `F` y nulos → 1 por decisión de limpieza.

In [220]:
sharks_activity["Fatal Y/N"] = sharks_activity["Fatal Y/N"].replace({"Y" : 0, "N": 1, "Nq": 1, "F": 1})

In [221]:
sharks_activity["Fatal Y/N"] = sharks_activity["Fatal Y/N"].fillna(1)

In [222]:
sharks_activity["Fatal Y/N"].unique()

array([1, 0], dtype=object)

### 2.6. Nombres de columnas

Renombramos las variables en minúsculas y convertimos `Fatal Y/N` en `life`. Guardamos el resultado en `sharks_mortal` para continuar con especies, horas y fechas.

In [223]:
sharks_mortal = sharks_activity.rename(columns={
    "Date" : "date",
    "Year" : "year",
    "Type" : "type",
    "Country" : "country",
    "State" : "state",
    "Activity" : "activity",
    "Sex" : "sex",
    "Age": "age",
    "Fatal Y/N" : "life",
    "Species " : "species",
    "Time" : "time"})

**Lectura del resultado:** `life` usa 0 para fallecidos y 1 para supervivientes. Los nulos asignados a 1 forman parte de las cifras posteriores.

### 2.7. Especies

Quitamos diferencias de escritura y tamaño para agrupar descripciones de la misma especie.

Conservamos categorías separadas para identificación desconocida, varias especies posibles y otras especies nombradas.

In [224]:
import pandas as pd
import re

def clean_species(value):
    if pd.isna(value):
        return None

    value = str(value).lower().strip()
    value = re.sub(r"\s+", " ", value)

    if value in ["", "nan"]:
        return None

    value = re.sub(r"\bwfite\b", "white", value)
    value = re.sub(r"\bblactip\b", "blacktip", value)

    if "..." in value:
        return "unknown"

    if re.search(r"\bor\b", value):
        return "multiple possible species"

    categories = {
        r"\bwhite\b": "white shark",
        r"\bsand[- ]?tiger\b": "sandtiger shark",
        r"\btiger\b": "tiger shark",
        r"\bbull\b": "bull shark",
        r"\bbronze whaler\b": "bronze whaler",
        r"\bblacktip reef\b": "blacktip reef shark",
        r"\bblacktip\b": "blacktip shark",
        r"\bspinner\b": "spinner shark",
        r"\bwobbegong\b": "wobbegong shark",
        r"\b(galapagos|nurse|seven[- ]gill|7[- ]gill|whitetip reef)\b": "other named species"}

    for pattern, category in categories.items():
        if re.search(pattern, value):
            return category

    return "unknown"

In [225]:
sharks_mortal["species_group"] = (
    sharks_mortal["species"].apply(clean_species))

In [226]:
print(sharks_mortal["species"])

63                     Suspected Great White Shark
74                                  Bronze whaler?
77                               Great White Shark
80                         5m (16.5ft) Great White
83                           Suspected Great White
                           ...                    
2825                 1.8 m [6'] grey-colored shark
2843    Blacktip shark, 1.2 m to 1.8 m [4' to 6'] 
2857                              1.2 m [4'] shark
2859                      Tiger shark, 4 m [13'] ?
2869                               3 m [10'] shark
Name: species, Length: 629, dtype: str


**Elegir el valor de relleno:** contamos las categorías concretas y seleccionamos la más frecuente con `idxmax()`. No usamos la mediana porque las especies son categorías, no números.

In [227]:
identified_species = sharks_mortal.loc[(sharks_mortal["species_group"] != "unknown") &(sharks_mortal["species_group"] != "multiple possible species") & (sharks_mortal["species_group"] != "other named species")]

species_counts = identified_species["species_group"].value_counts()

print(species_counts)

species_group
white shark            93
tiger shark            35
bronze whaler          19
bull shark             15
blacktip shark         14
wobbegong shark         8
spinner shark           6
sandtiger shark         2
blacktip reef shark     1
Name: count, dtype: int64


In [228]:
most_common_species = species_counts.idxmax()

print(most_common_species)

white shark


**Rellenar y comprobar:** marcamos los nulos en `species_imputed` y los sustituimos por la categoría más frecuente. Las categorías `unknown` y `multiple possible species` no se rellenan.

In [229]:
sharks_mortal["species_imputed"] = (sharks_mortal["species_group"].isna())

In [230]:
sharks_mortal["species_clean"] = (sharks_mortal["species_group"].fillna(most_common_species))

In [231]:
sharks_mortal["species_clean"].value_counts(dropna=False)

species_clean
white shark                  350
unknown                      153
tiger shark                   35
multiple possible species     22
bronze whaler                 19
bull shark                    15
blacktip shark                14
wobbegong shark                8
spinner shark                  6
other named species            4
sandtiger shark                2
blacktip reef shark            1
Name: count, dtype: int64

In [232]:
sharks_mortal.drop(["species", "species_group", "species_imputed"], axis=1, inplace=True)

**Resultado de la limpieza de especies**

Las variantes quedan agrupadas y los nulos se rellenan con la categoría identificada más frecuente. `unknown` y `multiple possible species` se conservan.

La celda anterior elimina `species`, `species_group` y `species_imputed` de `sharks_mortal`.

### 2.8. Franjas horarias

Agrupamos los formatos de hora para facilitar la comparación:

| Franja | Hora del reloj |
|---|---|
| Mañana | 06:00–11:59 |
| Tarde | 12:00–19:59 |
| Noche | 20:00–05:59 |

Son intervalos del reloj, no límites de luz solar.

In [233]:
print(sharks_mortal["time"])

63      0800hrs
74      1330hrs
77      0930hrs
80      0730hrs
83      1630hrs
         ...   
2825        NaN
2843      16h00
2857      14h00
2859      19h30
2869        NaN
Name: time, Length: 629, dtype: str


In [234]:

import pandas as pd

import re

**Textos y aproximaciones:** el diccionario asigna las descripciones a una franja. `AM` se trata como mañana; `Sunset`, `Dusk` y `Evening`, como tarde. Son convenciones del proyecto, no horas exactas.

In [235]:


time_descriptions = {
    "morning": "mañana",
    "early morning": "mañana",
    "just before noon": "mañana",
    '"just before 11h00"': "mañana",

    "afternoon": "tarde",
    "after noon": "tarde",
    "early afternoon": "tarde",
    "late afternoon": "tarde",

    "night": "noche",

    
    "am": "mañana",
    "after 1200hr": "tarde",
    "sunset": "tarde",
    "sundown": "tarde",
    "just before sundown": "tarde",
    "dusk": "tarde",
    "evening": "tarde",

    
    "07h00 - 08h00": "mañana",
    "09h00 -10h00": "mañana",
    "sometime between 06h00 & 08hoo": "mañana",
    "14h00-15h00": "tarde"}

**Interpretar horas:** Regex reconoce formatos como `0800hrs`, `13h30` y `8:04 pm`. Si hay una hora concreta con una anotación, la hora tiene prioridad. Los formatos inválidos quedan pendientes.

In [236]:
def clean_time(value):

    if pd.isna(value):
        return None

    value = str(value).lower().strip()

    value = re.sub(r"\s+", " ", value)

    if value in time_descriptions:
        return time_descriptions[value]

    value = re.sub(r"\s*\(sunset\)", "", value)

    value = re.sub(r"\s*hrs?$", "", value)

    value = value.replace("h", ":")

    if re.fullmatch(r"\d{4}", value):
        value = value[:2] + ":" + value[2:]

    match = re.fullmatch(r"(\d{1,2}):(\d{2})\s*(am|pm)?",value)

    if match is None:
        return None

    hour = int(match.group(1))
    minute = int(match.group(2))
    period = match.group(3)

    if minute > 59:
        return None

    if period is not None:

        if hour < 1 or hour > 12:
            return None

        hour = hour % 12

        if period == "pm":
            hour += 12

    if hour > 23:
        return None

    if 6 <= hour < 12:
        return "mañana"

    elif 12 <= hour < 20:
        return "tarde"

    else:
        return "noche"

In [237]:
sharks_mortal["time_group"] = (sharks_mortal["time"].apply(clean_time))

print(sharks_mortal["time_group"].value_counts(dropna=False))

time_group
tarde     309
mañana    233
NaN        80
noche       7
Name: count, dtype: int64


**Completar los pendientes:** marcamos los casos sin franja y usamos la categoría más frecuente para rellenarlos.

In [238]:
sharks_mortal["time_imputed"] = (sharks_mortal["time_group"].isna())

In [239]:
time_counts = sharks_mortal["time_group"].value_counts()

if time_counts.empty:
    print("No hay horas clasificadas para calcular la franja más frecuente.")

else:
    most_common_time = time_counts.idxmax()

    sharks_mortal["time_group"] = (sharks_mortal["time_group"].fillna(most_common_time))

    print("Franja utilizada para rellenar:", most_common_time)

Franja utilizada para rellenar: tarde


In [240]:
print(sharks_mortal["time_group"].value_counts(dropna=False))

print("Registros rellenados:", sharks_mortal["time_imputed"].sum())

sharks_mortal[["time", "time_group", "time_imputed"]].head(10)

sharks_mortal.drop(["time_imputed", "time"], axis=1, inplace=True)

time_group
tarde     389
mañana    233
noche       7
Name: count, dtype: int64
Registros rellenados: 80


### 2.9. Fechas

Revisamos los formatos antes de convertirlos a fechas. Necesitamos conservar el mes correcto para asignar la estación.

In [241]:
print(sharks_mortal["date"])

63      12th December
74        7th October
77      6th September
80        18th August
83         7th August
            ...      
2825      16-Jul-2000
2843      10-Jun-2000
2857      24-Mar-2000
2859      14-Mar-2000
2869      01-Feb-2000
Name: date, Length: 629, dtype: str


In [242]:
for value in sharks_mortal["date"].unique():
    print(value)

12th December
7th October
6th September
18th August
7th August
18th July
6th July
31st May
10-mar-25
22-ene-25
2-ene-25
1-dic-24
25-oct-24
11-oct-24
23-jul-24
18-jul-24
23-jun-24
20-abr-24
04 Mar 2024
09-Jan-2024
30 Dec-2023
28 Dec-2023
25 Dec-2023
31 Oct-2023
25 Oct 2023
15 Oct 2023
13 Oct-2023
02 Oct-2023
12 Sep-2023
11 Sep-2023
04 Sep 2023
25 Aug 2023
31 Jul-2023
24 Jul-2023
14 Jul-2023
03 Jul-2023
21-May 2023
13-May-2023
22-Apr-2023
09-Apr-2023
19-Feb-2023
07-Dec-2022
31-Oct-2022
06-Oct-2022
02-Oct-2022
31-Aug-2022
20-Jul-2022
19-Jul-2022
13-Jul-2022
10-Jul-2022
03-Jul-2022
27-May-2022
15-Mar-2022
13-Mar-2022
08-Mar-2022
11-Feb-2022
01-Jan-2022
04-Dec-2021
04-Oct-2021
03-Oct-2021
09-Sep-2021
05-Sep-2021
21-Aug-2021
07-Aug-2021
27-Jul-2021
21-Jul-2021
19-Jul-2021
05-Jul-2021
28-Jun-2021
23-Jun-2021
18-May-2021
03-May-2021
30-Apr-2021
06-Feb-2021
03-Jan-2021
02-Jan-2021
09-Dec-2020
08-Dec-2020
06-Dec-2020
02-Nov-2020
29-Oct-2020
09-Oct-2020
04-Oct-2020
15-Sep-2020
07-Sep-2020
30-Aug-

**Reconocer meses:** el diccionario reúne meses en inglés y abreviaturas en español. Así podemos interpretar, por ejemplo, `December` y `dic`.

In [243]:
import pandas as pd
import re

months = {
    "jan": 1, "january": 1, "ene": 1,
    "feb": 2, "february": 2,
    "mar": 3, "march": 3,
    "apr": 4, "april": 4, "abr": 4,
    "may": 5,
    "jun": 6, "june": 6,
    "jul": 7, "july": 7,
    "aug": 8, "august": 8, "ago": 8,
    "sep": 9, "sept": 9, "september": 9,
    "oct": 10, "october": 10,
    "nov": 11, "november": 11,
    "dec": 12, "december": 12, "dic": 12
}

**Reglas de conversión**

- Fecha completa: conservamos día, mes y año.
- Día y mes sin año: recuperamos el año de `year`.
- Mes y año: asignamos el día 15; no cambia la estación.
- Año incompleto: usamos `year` solo si es compatible.
- Fecha imposible, contradictoria o sin mes: dejamos `NaT`.
- `Reported`: marcamos que es fecha de notificación.

La función devuelve una fecha y un estado de revisión. No rellenamos meses desconocidos con una fecha mediana.

In [244]:
def clean_date(row):

   
    value = row["date"]

    try:
        year_number = float(str(row["year"]).replace(",", "."))

        if year_number.is_integer() and 1900 <= year_number <= 2026:
            reference_year = int(year_number)
        else:
            reference_year = None

    except (ValueError, TypeError):
        reference_year = None

    if pd.isna(value):
        return pd.NaT, "fecha ausente"

    text = str(value).lower().strip()

    reported = text.startswith("reported")
    text = re.sub(r"^reported\s*", "", text)

    text = re.sub(r"(\d+)(st|nd|rd|th)\b", r"\1", text)

    parts = re.findall(r"\d+|[a-z]+", text)

    status = "fecha completa"

    if (
        len(parts) == 2
        and parts[0] in months
        and parts[1].isdigit()
        and len(parts[1]) == 4
    ):
        day = 15
        month = months[parts[0]]
        year = int(parts[1])
        status = "día estimado: 15"

    elif (
        len(parts) in [2, 3]
        and parts[0].isdigit()
        and parts[1] in months
    ):
        day = int(parts[0])
        month = months[parts[1]]


        if len(parts) == 2:
            if reference_year is None:
                return pd.NaT, "falta el año"

            year = reference_year
            status = "año recuperado de year"

        else:
            year_text = parts[2]

            if not year_text.isdigit():
                return pd.NaT, "año no válido"

            if len(year_text) == 2:
                year = 2000 + int(year_text)

            elif len(year_text) == 4:
                year = int(year_text)

            elif (
                len(year_text) == 3
                and reference_year is not None
                and str(reference_year).startswith(year_text)
            ):
                year = reference_year
                status = "año incompleto recuperado de year"

            else:
                return pd.NaT, "año por revisar"

    else:
        return pd.NaT, "fecha incompleta o no reconocida"

    if reference_year is not None and year != reference_year:
        return pd.NaT, "fecha y year no coinciden"

    date = pd.to_datetime(
        f"{year}-{month:02d}-{day:02d}",
        format="%Y-%m-%d",
        errors="coerce")

    if pd.isna(date):
        return pd.NaT, "fecha imposible"

    if reported:
        status = "fecha de notificación; " + status

    return date, status

**Guardar el resultado:** `apply(..., axis=1)` permite leer fecha y año de la misma fila. Separamos la fecha limpia en `date_clean` y la explicación en `date_status`.

In [245]:
date_results = sharks_mortal.apply(clean_date, axis=1)

sharks_mortal["date_clean"] = pd.to_datetime(date_results.apply(lambda result: result[0]))

sharks_mortal["date_status"] = date_results.apply(lambda result: result[1])

sharks_mortal[["date", "year", "date_clean", "date_status"]].head(15)

,date,year,date_clean,date_status
63,12th December,2025.0,2025-12-12,año recuperado de year
74,7th October,2025.0,2025-10-07,año recuperado de year
77,6th September,2025.0,2025-09-06,año recuperado de year
80,18th August,2025.0,2025-08-18,año recuperado de year
83,7th August,2025.0,2025-08-07,año recuperado de year
90,18th July,2025.0,2025-07-18,año recuperado de year
93,6th July,2025.0,2025-07-06,año recuperado de year
100,31st May,2025.0,2025-05-31,año recuperado de year
111,10-mar-25,2025.0,2025-03-10,fecha completa
123,22-ene-25,2025.0,2025-01-22,fecha completa


### 2.10. Estaciones por país

Usamos estaciones por meses, con calendarios opuestos:

| Meses | USA | Australia |
|---|---|---|
| Diciembre–febrero | Invierno | Verano |
| Marzo–mayo | Primavera | Otoño |
| Junio–agosto | Verano | Invierno |
| Septiembre–noviembre | Otoño | Primavera |

Las fechas ausentes o de notificación quedan con estación `desconocida`.

In [246]:
def get_season(row):

    date = row["date_clean"]

    if pd.isna(date):
        return "desconocida"

    if "notificación" in row["date_status"]:
        return "desconocida"

    country = str(row["country"]).strip().lower()

    month = date.month

    if country in ["usa", "us", "united states", "united states of america"]:

        if month in [12, 1, 2]:
            return "invierno"
        elif month in [3, 4, 5]:
            return "primavera"
        elif month in [6, 7, 8]:
            return "verano"
        else:
            return "otoño"

    elif country == "australia":

        if month in [12, 1, 2]:
            return "verano"
        elif month in [3, 4, 5]:
            return "otoño"
        elif month in [6, 7, 8]:
            return "invierno"
        else:
            return "primavera"

    return "desconocida"

In [247]:
sharks_mortal["season"] = sharks_mortal.apply(
    get_season,
    axis=1)

print(sharks_mortal["date_status"].value_counts())

print(sharks_mortal.groupby(["country", "season"]).size())

sharks_mortal.loc[sharks_mortal["date_clean"].isna(),["date", "year", "date_status"]]

date_status
fecha completa                           609
año recuperado de year                     8
fecha de notificación; fecha completa      7
día estimado: 15                           2
año incompleto recuperado de year          1
fecha y year no coinciden                  1
fecha incompleta o no reconocida           1
Name: count, dtype: int64
country    season     
australia  desconocida      4
           invierno        45
           otoño           42
           primavera       50
           verano          51
usa        desconocida      5
           invierno        40
           otoño          173
           primavera       96
           verano         123
dtype: int64


,date,year,date_status
703,18-Feb-2018,2019.0,fecha y year no coinciden
2503,2004,2004.0,fecha incompleta o no reconocida


In [248]:
sharks_mortal.head()

,date,year,type,country,state,activity,sex,age,life,species_clean,time_group,date_clean,date_status,season
63,12th December,2025.0,unprovoked,usa,california,surfing,male,29,1,white shark,mañana,2025-12-12,año recuperado de year,invierno
74,7th October,2025.0,unprovoked,australia,south australia,surfing,male,29,1,bronze whaler,tarde,2025-10-07,año recuperado de year,primavera
77,6th September,2025.0,unprovoked,australia,new south wales,surfing,male,57,0,white shark,mañana,2025-09-06,año recuperado de year,primavera
80,18th August,2025.0,unprovoked,australia,new south wales,surfing,male,29,1,white shark,mañana,2025-08-18,año recuperado de year,invierno
83,7th August,2025.0,unprovoked,australia,new south wales,surfing,male,9,1,white shark,tarde,2025-08-07,año recuperado de year,invierno


### 2.11. Preparar la tabla de análisis

Eliminamos la fecha original y su estado de revisión de `sharks_mortal`. Después seleccionamos las 12 columnas finales en un orden fácil de consultar.

Esta simplificación facilita la lectura, pero retira información útil para revisar las estimaciones.

In [249]:
sharks_mortal.drop(["date_status", "date"], axis=1, inplace=True)

In [250]:
sharks_limpio = sharks_mortal[["date_clean", "season", "year","time_group", "country", "state", "sex", "age", "activity", "type","species_clean","life"]]

In [251]:
sharks_limpio.head()

,date_clean,season,year,time_group,country,state,sex,age,activity,type,species_clean,life
63,2025-12-12,invierno,2025.0,mañana,usa,california,male,29,surfing,unprovoked,white shark,1
74,2025-10-07,primavera,2025.0,tarde,australia,south australia,male,29,surfing,unprovoked,bronze whaler,1
77,2025-09-06,primavera,2025.0,mañana,australia,new south wales,male,57,surfing,unprovoked,white shark,0
80,2025-08-18,invierno,2025.0,mañana,australia,new south wales,male,29,surfing,unprovoked,white shark,1
83,2025-08-07,invierno,2025.0,tarde,australia,new south wales,male,9,surfing,unprovoked,white shark,1


**Resultado:** `sharks_limpio` contiene las 12 variables seleccionadas para las mediciones. 

## 3. Comprobación de la tabla final

Revisamos el tamaño, los países, los límites del periodo y las categorías de actividad y tipo de incidente.

In [252]:
print("Filas y columnas:", sharks_limpio.shape)

print(sharks_limpio["country"].unique())

print(sharks_limpio["year"].min())
print(sharks_limpio["year"].max())

print(sharks_limpio["type"].unique())
print(sharks_limpio["activity"].unique())

Filas y columnas: (629, 12)
<ArrowStringArray>
['usa', 'australia']
Length: 2, dtype: str
2000.0
2025.0
<ArrowStringArray>
['unprovoked']
Length: 1, dtype: str
<ArrowStringArray>
['surfing']
Length: 1, dtype: str


### Lectura de la comprobación

La salida guardada muestra **629 registros y 12 columnas**, años entre 2000 y 2025, países `usa` y `australia`, tipo `unprovoked` y actividad `surfing`.

Estas etiquetas incorporan las decisiones anteriores: el filtro de tipo incluye casos dudosos reclasificados. No se muestra una comprobación de duplicados en este notebook.

## 4. Análisis de los incidentes

### 4.1. Distribución por país

Contamos las filas de cada país y calculamos su porcentaje sobre los **629 registros**.

In [253]:
attacks_by_country = sharks_limpio["country"].value_counts()
print(attacks_by_country)

country
usa          437
australia    192
Name: count, dtype: int64


In [254]:
total_attacks = attacks_by_country.sum()

In [255]:
percentage_by_country = (attacks_by_country / total_attacks * 100).round(2)
print(percentage_by_country)

country
usa          69.48
australia    30.52
Name: count, dtype: float64


#### Conclusión

- **USA:** 437 incidentes, el **69,48 %** del total.
- **Australia:** 192 incidentes, el **30,52 %**.

USA concentra más del doble de registros. Esto no demuestra mayor riesgo: desconocemos cuántas personas practican la actividad y su tiempo de exposición.

### 4.2. Distribución por estado

Queremos comparar el peso de cada estado dentro de su país.

In [256]:
estado = sharks_limpio.groupby("country")["state"].value_counts(normalize=True) * 100

porcentaje_estado = (estado / estado.sum()) * 100

porcentaje_estado

country    state            
australia  new south wales      27.083333
           western australia    10.937500
           south australia       4.687500
           victoria              3.906250
           queensland            2.604167
           tasmania              0.781250
usa        florida              30.434783
           california            6.636156
           hawaii                6.636156
           north carolina        2.059497
           oregon                1.716247
           south carolina        0.800915
           texas                 0.572082
           new york              0.343249
           georgia               0.228833
           new jersey            0.114416
           guam                  0.114416
           rhode island          0.114416
           washington            0.114416
           virginia              0.114416
Name: proportion, dtype: float64

#### Conclusión

Florida encabeza los registros de USA y New South Wales los de Australia. Son los estados por país según el contexto estudiado donde se producen más ataques de tiburón.

### 4.3. Distribución por franja horaria

Contamos mañana, tarde y noche. Cada porcentaje se calcula sobre el total de registros de ambos países, no dentro de cada país.

In [257]:
ataques = sharks_limpio.groupby("time_group").size()

porcentaje = (ataques / len(sharks_limpio)) * 100

porcentaje

time_group
mañana    37.042925
noche      1.112878
tarde     61.844197
dtype: float64

#### Conclusión

- **Tarde:** 61,84 % de los registros.
- **Mañana:** 37,04 %.
- **Noche:** 1,11 %.

### 4.4. Mortalidad por franja horaria

Contamos supervivientes y fallecidos por franja. Después calculamos los porcentajes **dentro de cada franja**.

Recordatorio: `life = 1` es superviviente y `life = 0` es fallecido.

In [258]:
sharks_limpio.groupby(["time_group", "life"]).size()

time_group  life
mañana      0        16
            1       217
noche       1         7
tarde       0        11
            1       378
dtype: int64

In [259]:
sharks_limpio.groupby("time_group")["life"].value_counts(normalize=True) * 100

time_group  life
mañana      1        93.133047
            0         6.866953
noche       1       100.000000
tarde       1        97.172237
            0         2.827763
Name: proportion, dtype: float64

#### Conclusión

| Franja | Incidentes | Fallecidos | Mortalidad |
|---|---:|---:|---:|
| Mañana | 233 | 16 | 6,87 % |
| Tarde | 389 | 11 | 2,83 % |
| Noche | 7 | 0 | 0,00 % |

La mañana tiene la mayor mortalidad proporcional de esta tabla. Por la noche **sobrevivieron los siete casos registrados**; no hubo fallecidos.

Siete casos son pocos para establecer un patrón. Los valores rellenados de hora y supervivencia también afectan a esta comparación.

### 4.5. Edad y sexo

Contamos cada combinación de edad y sexo. El porcentaje posterior muestra el peso de cada edad **dentro de su grupo de sexo**.

In [260]:
sharks_limpio.groupby(["age", "sex"]).size()

age  sex   
9    female    1
     male      4
10   male      1
11   male      2
12   female    1
              ..
63   male      1
64   female    1
     male      1
65   male      1
68   male      1
Length: 78, dtype: int64

In [261]:
porcentaje_sexoedad = sharks_limpio.groupby("sex")["age"].value_counts(normalize=True) * 100
porcentaje_sexoedad


sex     age
female  29     34.482759
        15      6.896552
        35      6.896552
        17      6.896552
        13      6.896552
                 ...    
male    10      0.175131
        34      0.175131
        53      0.175131
        63      0.175131
        68      0.175131
Name: proportion, Length: 78, dtype: float64

#### Conclusión

Los 29 años son la edad con mayor proporción dentro de ambos grupos: **34,48 % en mujeres** y **22,77 % en hombres**.

### 4.6. Estaciones y países

El código calcula qué porcentaje de los incidentes de **cada estación** corresponde a cada país.

No calcula qué estación concentra más incidentes dentro de USA o Australia. Esa segunda comparación es la necesaria para contrastar nuestra hipótesis.

In [262]:
temporada = sharks_limpio.groupby("season").size()

porcentaje_temporada = sharks_limpio.groupby("season")["country"].value_counts(normalize=True) * 100

porcentaje_temporada

season       country  
desconocida  usa          55.555556
             australia    44.444444
invierno     australia    52.941176
             usa          47.058824
otoño        usa          80.465116
             australia    19.534884
primavera    usa          65.753425
             australia    34.246575
verano       usa          70.689655
             australia    29.310345
Name: proportion, dtype: float64

In [263]:
incidentes_estacion = (sharks_limpio.groupby("country")["season"].value_counts())
porcentaje_estacion = (sharks_limpio.groupby("country")["season"].value_counts(normalize=True).mul(100).round(2))
print("Incidentes por estación y país:")
print(incidentes_estacion)
print("\nPorcentajes dentro de cada país:")
print(porcentaje_estacion)


Incidentes por estación y país:
country    season     
australia  verano          51
           primavera       50
           invierno        45
           otoño           42
           desconocida      4
usa        otoño          173
           verano         123
           primavera       96
           invierno        40
           desconocida      5
Name: count, dtype: int64

Porcentajes dentro de cada país:
country    season     
australia  verano         26.56
           primavera      26.04
           invierno       23.44
           otoño          21.88
           desconocida     2.08
usa        otoño          39.59
           verano         28.15
           primavera      21.97
           invierno        9.15
           desconocida     1.14
Name: proportion, dtype: float64


### Conclusión — Distribución de incidentes por estación y país

Primero comparamos qué país aporta más registros dentro de cada estación. Observamos que USA representa el 65,75 % en primavera, el 70,69 % en verano y el 80,47 % en otoño. En invierno,Australia reúne el 52,94 % y USA el 47,06 %.

Después comparamos las estaciones dentro de cada país:

- En **Australia**, encontramos más registros en verano:51 incidentes (26,56 %). Sin embargo, la diferencia con primavera es mínima: 50 incidentes (26,04 %).
- En **USA**, encontramos más registros en otoño: 173 incidentes (39,59 %), seguido del verano, con 123 incidentes (28,15 %).

**Nuestra hipótesis no se cumple en ambos países.** En Australia observamos un máximo en verano, aunque solo supera a primavera en un incidente. En USA, el máximo corresponde al otoño.

### 4.7. Distribución anual

Contamos los registros por año y dividimos entre el total del periodo. Así obtenemos el peso de cada año en el conjunto de 2000–2025.

In [264]:
año = sharks_limpio.groupby("year").size()

porcentaje_año = (año / año.sum()) * 100

porcentaje_año


year
2000.0    2.225755
2001.0    4.451510
2002.0    3.179650
2003.0    4.769475
2004.0    2.702703
2005.0    3.974563
2006.0    3.497615
2007.0    4.928458
2008.0    4.451510
2009.0    4.769475
2010.0    3.656598
2011.0    3.338633
2012.0    5.882353
2013.0    4.133545
2014.0    4.451510
2015.0    4.451510
2016.0    5.246423
2017.0    4.928458
2018.0    3.974563
2019.0    4.292528
2020.0    4.610493
2021.0    3.020668
2022.0    2.543720
2023.0    3.338633
2024.0    1.430843
2025.0    1.748808
dtype: float64

### 4.8. Mortalidad global y por país

Calculamos dos medidas:

- **Global:** fallecidos entre los 629 registros.
- **Por país:** fallecidos entre los registros de cada país.

Como `life` vale 1 para supervivientes, su media es la proporción de supervivencia. Restarla a 100, tras convertirla a porcentaje, da la mortalidad.

In [265]:
print(sharks_limpio["life"].value_counts(dropna=False))

life
1    602
0     27
Name: count, dtype: int64


In [266]:
fatal_attacks = sharks_limpio.loc[sharks_limpio["life"] == 0].shape[0]

total_attacks = sharks_limpio.shape[0]

fatal_percentage_total = round(fatal_attacks / total_attacks * 100, 2)

print("Incidentes mortales:", fatal_attacks)
print("Total de incidentes:", total_attacks)
print("Porcentaje mortal:", fatal_percentage_total, "%")

Incidentes mortales: 27
Total de incidentes: 629
Porcentaje mortal: 4.29 %


In [267]:
life_by_country = sharks_limpio.groupby(["country", "life"]).size()
print(life_by_country)


country    life
australia  0        21
           1       171
usa        0         6
           1       431
dtype: int64


In [268]:
survival_percentage = (sharks_limpio.groupby("country")["life"].mean() * 100)
print(survival_percentage)

country
australia      89.0625
usa          98.627002
Name: life, dtype: object


In [269]:
fatal_percentage = 100 - survival_percentage
print(fatal_percentage)

country
australia     10.9375
usa          1.372998
Name: life, dtype: object


In [270]:
print("Porcentaje de supervivientes:")
print(survival_percentage.round(2))

Porcentaje de supervivientes:
country
australia    89.06
usa          98.63
Name: life, dtype: object


In [271]:
print("Porcentaje de fallecidos:")
print(fatal_percentage.round(2))

Porcentaje de fallecidos:
country
australia    10.94
usa           1.37
Name: life, dtype: object


#### Conclusión

De los **629 registros**, 27 fueron mortales (**4,29 %**) y 602 no mortales (**95,71 %**).

| País | Incidentes | Fallecidos | Mortalidad |
|---|---:|---:|---:|
| Australia | 192 | 21 | 10,94 % |
| USA | 437 | 6 | 1,37 % |

USA acumula más incidentes, pero Australia tiene más fallecidos y mayor mortalidad entre los casos analizados.

## 5. Análisis de especies

Comparamos categorías, distribución geográfica y mortalidad usando la tabla final.

### 5.1. Incidentes por categoría

Contamos los registros de cada categoría y los dividimos entre los 629 casos. Este porcentaje muestra su peso en el total de la tabla.

In [272]:
import pandas as pd

species_counts = sharks_limpio["species_clean"].value_counts(dropna=False)

species_summary = pd.DataFrame({"incidentes": species_counts})

species_summary["porcentaje_total"] = (species_summary["incidentes"]/ sharks_limpio.shape[0]* 100).round(2)

species_summary


,incidentes,porcentaje_total
species_clean,,
white shark,350,55.64
unknown,153,24.32
tiger shark,35,5.56
multiple possible species,22,3.50
bronze whaler,19,3.02
bull shark,15,2.38
blacktip shark,14,2.23
wobbegong shark,8,1.27
spinner shark,6,0.95


#### Conclusión

- **White shark:** 350 registros (55,64 %).
- **Tiger shark:** 35 (5,56 %).
- **Bronze whaler:** 19 (3,02 %).

`unknown` y `multiple possible species` suman 175 casos (**27,82 %**). El predominio de white shark incluye los nulos asignados durante la limpieza.

### 5.2. Especies por país

Contamos las categorías dentro de cada país. El denominador es el total nacional: **192 en Australia** y **437 en USA**.

`map()` incorpora ese total a cada fila para calcular los porcentajes.

In [273]:
country_totals = sharks_limpio["country"].value_counts()

species_country = sharks_limpio.groupby(["country", "species_clean"]).size().reset_index(name="incidentes")

species_country["total_pais"] = (species_country["country"].map(country_totals))

species_country["porcentaje_en_pais"] = (species_country["incidentes"]/ species_country["total_pais"]* 100).round(2)

species_country = species_country.sort_values(["country", "incidentes"],ascending=[True, False])

print(species_country.to_string(index=False))

  country             species_clean  incidentes  total_pais  porcentaje_en_pais
australia               white shark         114         192               59.38
australia                   unknown          38         192               19.79
australia             bronze whaler          19         192                9.90
australia           wobbegong shark           8         192                4.17
australia                bull shark           5         192                2.60
australia               tiger shark           4         192                2.08
australia multiple possible species           3         192                1.56
australia       other named species           1         192                0.52
      usa               white shark         236         437               54.00
      usa                   unknown         115         437               26.32
      usa               tiger shark          31         437                7.09
      usa multiple possible species     

#### Conclusión

White shark representa **114 registros en Australia (59,38 %)** y **236 en USA (54,00 %)**.

Entre las otras categorías concretas destacan:

- **Australia:** bronze whaler, 19 registros (9,90 %), y wobbegong shark, 8 (4,17 %).
- **USA:** tiger shark, 31 (7,09 %), y blacktip shark, 14 (3,20 %).

### 5.3. Especies por estado

Unimos los recuentos de país, estado y especie con el total de cada estado mediante `merge()`.

**Porcentaje:** incidentes de la categoría en el estado / total de incidentes del estado × 100.

In [274]:

state_totals = sharks_limpio.groupby(["country", "state"]).size().reset_index(name="total_estado")


species_state = sharks_limpio.groupby(["country", "state", "species_clean"]).size().reset_index(name="incidentes")


species_state = pd.merge(species_state,state_totals,on=["country", "state"],how="left")


species_state["porcentaje_en_estado"] = (species_state["incidentes"]/ species_state["total_estado"]* 100).round(2)

species_state = species_state.sort_values(["country", "state", "incidentes"],ascending=[True, True, False])

print(species_state.to_string(index=False))

print(
    "Registros sin estado:",
    sharks_limpio["state"].isna().sum()
)

  country             state             species_clean  incidentes  total_estado  porcentaje_en_estado
australia   new south wales               white shark          57           104                 54.81
australia   new south wales                   unknown          23           104                 22.12
australia   new south wales             bronze whaler          10           104                  9.62
australia   new south wales           wobbegong shark           6           104                  5.77
australia   new south wales                bull shark           4           104                  3.85
australia   new south wales               tiger shark           3           104                  2.88
australia   new south wales multiple possible species           1           104                  0.96
australia        queensland               white shark           6            10                 60.00
australia        queensland                   unknown           2            10   

#### Conclusión

- **New South Wales:** 104 incidentes; white shark supone el 54,81 %.
- **Western Australia:** 42 incidentes; white shark supone el 66,67 %.
- **Florida:** 266 incidentes; white shark supone el 52,26 % y unknown el 31,20 %.
- **California:** 58 incidentes; white shark supone el 86,21 %.
- **Hawaii:** 58 incidentes; tiger shark supone el 53,45 %.

Los 31 registros estadounidenses de tiger shark se concentran en Hawaii.

### 5.4. Mortalidad por especie

Calculamos incidentes con resultado conocido, supervivientes y fallecidos.

- **Mortalidad:** fallecidos / incidentes de la categoría × 100.
- **Peso en los fallecidos:** fallecidos de la categoría / los 27 fallecidos totales × 100.

Mostramos dos ordenaciones porque más fallecidos no siempre significa mayor porcentaje de mortalidad.

In [275]:
species_mortality = sharks_limpio.groupby("species_clean")["life"].agg(["count", "sum"])

species_mortality = species_mortality.rename(columns={"count": "incidentes_con_resultado","sum": "supervivientes"})


species_mortality["fallecidos"] = (species_mortality["incidentes_con_resultado"]- species_mortality["supervivientes"])

species_mortality["porcentaje_mortalidad"] = (species_mortality["fallecidos"]/ species_mortality["incidentes_con_resultado"]* 100).round(2)

total_deaths = sharks_limpio.loc[sharks_limpio["life"] == 0].shape[0]

if total_deaths > 0:
    species_mortality["porcentaje_de_fallecidos_totales"] = (species_mortality["fallecidos"]/ total_deaths* 100).round(2)
else:
    species_mortality["porcentaje_de_fallecidos_totales"] = float("nan")

print("Orden por número de fallecidos:")
print(species_mortality.sort_values("fallecidos", ascending=False).to_string())

print("Orden por porcentaje de mortalidad:")
print(species_mortality.sort_values("porcentaje_mortalidad", ascending=False).to_string())

Orden por número de fallecidos:
                           incidentes_con_resultado supervivientes fallecidos porcentaje_mortalidad porcentaje_de_fallecidos_totales
species_clean                                                                                                                       
white shark                                     350            328         22                  6.29                            81.48
unknown                                         153            150          3                  1.96                            11.11
tiger shark                                      35             33          2                  5.71                             7.41
blacktip reef shark                               1              1          0                   0.0                              0.0
blacktip shark                                   14             14          0                   0.0                              0.0
bronze whaler                        

#### Conclusión

| Categoría | Fallecidos | Mortalidad en la categoría | Peso en los 27 fallecidos |
|---|---:|---:|---:|
| White shark | 22 | 6,29 % | 81,48 % |
| Tiger shark | 2 | 5,71 % | 7,41 % |
| Unknown | 3 | 1,96 % | 11,11 % |

Observamos con White Shark en el rango estudiado es el tiburón con mayor número de fallecidos y el que más peso tiene en el índice global.

### 5.5. Mortalidad por especie y país

Separamos las categorías por país y calculamos:

- Mortalidad entre los incidentes de esa categoría en ese país.
- Peso de la categoría en los fallecidos del país: **21 en Australia** y **6 en USA**.

In [276]:
country_species_mortality = sharks_limpio.groupby(["country", "species_clean"])["life"].agg(["count", "sum"]).reset_index()

country_species_mortality = country_species_mortality.rename(
    columns={"count": "incidentes_con_resultado","sum": "supervivientes"})


country_species_mortality["fallecidos"] = (country_species_mortality["incidentes_con_resultado"]- country_species_mortality["supervivientes"])

country_species_mortality["porcentaje_mortalidad"] = (country_species_mortality["fallecidos"]/ country_species_mortality["incidentes_con_resultado"] * 100).round(2)

deaths_by_country = sharks_limpio.loc[ sharks_limpio["life"] == 0].groupby("country").size()

country_species_mortality["fallecidos_totales_pais"] = (country_species_mortality["country"].map(deaths_by_country))

country_species_mortality["porcentaje_de_fallecidos_del_pais"] = (country_species_mortality["fallecidos"]/ country_species_mortality["fallecidos_totales_pais"]* 100).round(2)

country_species_mortality = country_species_mortality.sort_values(["country", "fallecidos"],ascending=[True, False])

print(country_species_mortality.to_string(index=False))

  country             species_clean  incidentes_con_resultado supervivientes fallecidos porcentaje_mortalidad  fallecidos_totales_pais porcentaje_de_fallecidos_del_pais
australia               white shark                       114             96         18                 15.79                       21                             85.71
australia                   unknown                        38             35          3                  7.89                       21                             14.29
australia             bronze whaler                        19             19          0                   0.0                       21                               0.0
australia                bull shark                         5              5          0                   0.0                       21                               0.0
australia multiple possible species                         3              3          0                   0.0                       21                     

#### Conclusión

- **Australia — white shark:** 18 fallecidos entre 114 incidentes (15,79 %). Representa el 85,71 % de los fallecidos del país.
- **USA — white shark:** 4 fallecidos entre 236 incidentes (1,69 %). Representa el 66,67 % de los fallecidos del país.
- **USA — tiger shark:** 2 fallecidos entre 31 incidentes (6,45 %). Representa el 33,33 % de los fallecidos del país.

Los otros tres fallecidos australianos están en `unknown`. Tiger shark no presenta fallecidos en Australia, pero solo tiene cuatro registros.

En USA, white shark acumula más fallecidos y tiger shark tiene mayor mortalidad proporcional.

## Conclusiones generales y propuesta de aplicación

### 1. Identificamos dónde empezar

Registramos 437 incidentes en USA, el 69,48 % del total, y 192 en Australia, el 30,52 %.

Dentro de cada país, encontramos la mayor concentración en Florida, con 266 registros, y New South Wales, con 104.
Seleccionamos ambos estados como puntos de partida para presentar nuestra propuesta a escuelas de surf.

### 2. Adaptamos el calendario a cada país

En USA, encontramos más incidentes en otoño: 173 casos, el 39,59 % de sus registros.

En Australia, observamos cifras muy similares en verano, con 51 casos (26,56 %), y primavera, con 50 (26,04 %).

Por tanto, nuestra hipótesis de que el verano concentra más incidentes no se cumple en ambos países. Proponemos preparar las campañas antes del otoño estadounidense y antes de la primavera australiana, manteniéndolas durante el verano.

### 3. Distinguimos frecuencia y mortalidad

Concentramos el 61,84 % de los registros en la tarde. Sin embargo, encontramos mayor mortalidad proporcional por la mañana: 6,87 %, frente al 2,83 % de la tarde.

En el conjunto del análisis contamos 27 fallecidos, el 4,29 % de los incidentes. Australia presenta una mortalidad del 10,94 %, frente al 1,37 % de USA.

Concluimos que las zonas y franjas con más incidentes no coinciden necesariamente con las de mayor mortalidad. Por eso, incorporamos ambas medidas a nuestra propuesta.

### 4. Adaptamos los contenidos al territorio

Encontramos diferencias en las categorías de especies registradas. White shark representa el 55,64 % de los incidentes y reúne el 81,48 % de los fallecidos de la tabla analizada.

En Hawaii destaca tiger shark, con el 53,45 % de los registrosdel estado. Utilizamos estas diferencias para preparar contenidos informativos específicos para cada zona.

Completamos el panel con la evolución anual y los perfiles de edad y sexo. Interpretamos estos últimos teniendo en cuenta las edades y categorías que hemos rellenado durante la limpieza.

### 5. Proponemos un servicio para escuelas de surf

Nuestra propuesta combina tres elementos:

- Un panel para consultar los incidentes por territorio,
  estación, horario y especie.
- Un calendario de comunicación para planificar la información
  dirigida a alumnos y monitores.
- Formación ante emergencias impartida por profesionales
  cualificados.

Los resultados respaldan nuestra hipótesis principal: encontramos diferencias territoriales, estacionales y de mortalidad que dan sentido a una propuesta adaptada a cada zona.

Como siguiente paso, proponemos presentar el servicio a escuelas de Florida y New South Wales, conocer sus necesidades y comprobarsu interés antes de desarrollar una primera versión.